In [5]:
import requests
import zipfile
import os

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "../data/ml-latest-small.zip"
extract_path = "../data/"

os.makedirs(extract_path, exist_ok=True)

response = requests.get(url, verify=False)

with open(zip_path, "wb") as f:
    f.write(response.content)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("../data")

print("Downloaded and extracted.")

C:\Users\tyuce\recommendation_system\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'files.grouplens.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Downloaded and extracted.


In [6]:
print(os.listdir("../data"))

['ml-latest-small', 'ml-latest-small.zip']


In [7]:
print(os.listdir("../data/ml-latest-small"))

['links.csv', 'movies.csv', 'ratings.csv', 'README.txt', 'tags.csv']


In [10]:
import pandas as pd

ratings = pd.read_csv("../data/ml-latest-small/ratings.csv")
movies = pd.read_csv("../data/ml-latest-small/movies.csv")

In [12]:
print(ratings.head(),"\n", movies.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931 
    movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [13]:
print("Ratings shape:", ratings.shape)
print("Movies shape:", movies.shape)

print("\nRatings columns:", ratings.columns.tolist())
print("Movies columns:", movies.columns.tolist())

Ratings shape: (100836, 4)
Movies shape: (9742, 3)

Ratings columns: ['userId', 'movieId', 'rating', 'timestamp']
Movies columns: ['movieId', 'title', 'genres']


In [14]:
print("Unique users:", ratings["userId"].nunique())
print("Unique rated movies:", ratings["movieId"].nunique())

Unique users: 610
Unique rated movies: 9724


In [15]:
print(ratings["rating"].describe())

count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64


In [16]:
print("\nMissing values in ratings:")
print(ratings.isna().sum())

print("\nMissing values in movies:")
print(movies.isna().sum())


Missing values in ratings:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Missing values in movies:
movieId    0
title      0
genres     0
dtype: int64


In [17]:
ratings_per_movie = (
    ratings.groupby("movieId")
    .size()
    .sort_values(ascending=False)
)

print(ratings_per_movie.head(10))

movieId
356     329
318     317
296     307
593     279
2571    278
260     251
480     238
110     237
589     224
527     220
dtype: int64


In [18]:
user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(movie_ids)
}

In [19]:
ratings["user_idx"] = ratings["userId"].map(user_to_idx)
ratings["movie_idx"] = ratings["movieId"].map(movie_to_idx)

print(ratings.head())

   userId  movieId  rating  timestamp  user_idx  movie_idx
0       1        1     4.0  964982703         0          0
1       1        3     4.0  964981247         0          1
2       1        6     4.0  964982224         0          2
3       1       47     5.0  964983815         0          3
4       1       50     5.0  964982931         0          4


In [22]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 80668
Test size: 20168


In [24]:
import numpy as np

num_users = ratings["user_idx"].nunique()
num_movies = ratings["movie_idx"].nunique()

num_features = 20

np.random.seed(42)

U = np.random.normal(
    0,
    0.1,
    size=(num_users, num_features)
)

V = np.random.normal(
    0,
    0.1,
    size=(num_movies, num_features)
)

user_bias = np.zeros(num_users)
movie_bias = np.zeros(num_movies)

global_mean = train_df["rating"].mean()

print("U shape:", U.shape)
print("V shape:", V.shape)
print("Global mean:", global_mean)

U shape: (610, 20)
V shape: (9724, 20)
Global mean: 3.502572271532702


In [25]:
lr = 0.01
reg = 0.02
epochs = 10

for epoch in range(epochs):

    # shuffle training rows every epoch
    shuffled = train_df.sample(frac=1, random_state=epoch)

    for row in shuffled.itertuples(index=False):

        u = row.user_idx
        i = row.movie_idx
        rating = row.rating

        prediction = (
            global_mean
            + user_bias[u]
            + movie_bias[i]
            + U[u] @ V[i]
        )

        error = rating - prediction

        user_vec = U[u].copy()
        movie_vec = V[i].copy()

        user_bias[u] += lr * (
            error - reg * user_bias[u]
        )

        movie_bias[i] += lr * (
            error - reg * movie_bias[i]
        )

        U[u] += lr * (
            error * movie_vec
            - reg * user_vec
        )

        V[i] += lr * (
            error * user_vec
            - reg * movie_vec
        )

    print(f"Epoch {epoch + 1} finished")

Epoch 1 finished
Epoch 2 finished
Epoch 3 finished
Epoch 4 finished
Epoch 5 finished
Epoch 6 finished
Epoch 7 finished
Epoch 8 finished
Epoch 9 finished
Epoch 10 finished


In [33]:
def predict_rating(u, i, clip=True):
    prediction = (
        global_mean
        + user_bias[u]
        + movie_bias[i]
        + U[u] @ V[i]
    )

    if clip:
        return np.clip(prediction, 0.5, 5.0)

    return prediction

In [27]:
train_predictions = []

for row in train_df.itertuples(index=False):
    pred = predict_rating(row.user_idx, row.movie_idx)
    train_predictions.append(pred)

train_predictions = np.array(train_predictions)

train_rmse = np.sqrt(
    np.mean(
        (train_df["rating"].to_numpy() - train_predictions) ** 2
    )
)

print("Train RMSE:", train_rmse)

Train RMSE: 0.7763292773441556


In [28]:
test_predictions = []

for row in test_df.itertuples(index=False):
    pred = predict_rating(row.user_idx, row.movie_idx)
    test_predictions.append(pred)

test_predictions = np.array(test_predictions)

test_rmse = np.sqrt(
    np.mean(
        (test_df["rating"].to_numpy() - test_predictions) ** 2
    )
)

print("Test RMSE:", test_rmse)

Test RMSE: 0.8814808658797543


In [38]:
def recommend_movies(user_id, k=10):
    user_idx = user_to_idx[user_id]

    rated_movie_ids = set(
        ratings.loc[
            ratings["userId"] == user_id,
            "movieId"
        ]
    )

    recommendations = []

    for movie_id in movie_ids:

        if movie_id in rated_movie_ids:
            continue

        movie_idx = movie_to_idx[movie_id]

        pred = predict_rating(
            user_idx,
            movie_idx,
            clip=False
        )

        recommendations.append(
            (movie_id, pred)
        )

    recommendations = sorted(
        recommendations,
        key=lambda x: x[1],
        reverse=True
    )

    top_k = recommendations[:k]

    result = []

    for movie_id, score in top_k:

        title = movies.loc[
            movies["movieId"] == movie_id,
            "title"
        ].iloc[0]

        result.append({
            "title": title,
            "predicted_score": round(float(score), 3)
        })

    return pd.DataFrame(result)

In [39]:
recommend_movies(1, k=10)

,title,predicted_score
0,Casablanca (1942),5.166
1,"Shawshank Redemption, The (1994)",5.160
2,Dr. Strangelove or: How I Learned to Stop Worr...,5.120
3,"Great Escape, The (1963)",5.112
4,Rear Window (1954),5.079
5,"Godfather: Part II, The (1974)",5.078
6,Cool Hand Luke (1967),5.068
7,"Manchurian Candidate, The (1962)",5.068
8,Touch of Evil (1958),5.061
9,Lawrence of Arabia (1962),5.041
